# VisProbe: Vanilla vs Adversarially-Trained Model Comparison

**GPU Runtime Required:** Runtime > Change runtime type > T4 GPU

**Dataset:** ImageNet validation set

Expected time: ~20-30 minutes total

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch torchvision
!pip install -q robustbench
!pip install -q git+https://github.com/fra31/auto-attack
!pip install -q adversarial-robustness-toolbox
!pip install -q gdown  # For downloading from Google Drive

print("Dependencies installed!")

In [ ]:
# Cell 2: Install VisProbe
# Option A: From GitHub (update with your actual repo URL)
!pip install -q git+https://github.com/bilgedemirkaya/VisProbe

# Option B: Upload zip file
# from google.colab import files
# import zipfile
# print("Upload VisProbe.zip")
# uploaded = files.upload()
# for filename in uploaded.keys():
#     if filename.endswith('.zip'):
#         with zipfile.ZipFile(filename, 'r') as zip_ref:
#             zip_ref.extractall('/content/')
# !pip install -q -e /content/VisProbe

print("VisProbe installed!")

In [ ]:
# Cell 3: Upload ImageNet validation data
# You need to upload your ImageNet validation set

import os
from google.colab import files
import zipfile

IMAGENET_PATH = "/content/imagenet_val"

if not os.path.exists(IMAGENET_PATH):
    print("="*60)
    print("IMAGENET VALIDATION DATA REQUIRED")
    print("="*60)
    print("\nPlease upload your ImageNet validation set as a zip file.")
    print("Expected structure inside zip:")
    print("  imagenet_val/")
    print("    n01440764/")
    print("      ILSVRC2012_val_00000293.JPEG")
    print("      ...")
    print("    n01443537/")
    print("      ...")
    print("\n")
    
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            print(f"\nExtracting {filename}...")
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/')
            print("Extraction complete!")
            
            # Find the extracted folder
            for item in os.listdir('/content/'):
                if 'imagenet' in item.lower() or 'val' in item.lower():
                    if os.path.isdir(f'/content/{item}'):
                        IMAGENET_PATH = f'/content/{item}'
                        break

# Verify the data
if os.path.exists(IMAGENET_PATH):
    num_classes = len([d for d in os.listdir(IMAGENET_PATH) if os.path.isdir(os.path.join(IMAGENET_PATH, d))])
    print(f"\nImageNet validation set found at: {IMAGENET_PATH}")
    print(f"Number of classes: {num_classes}")
else:
    raise FileNotFoundError(f"ImageNet data not found at {IMAGENET_PATH}")

In [ ]:
# Cell 4: Setup and load data
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ImageNet normalization and transforms
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

imagenet_val = ImageFolder(root=IMAGENET_PATH, transform=transform)
print(f"Loaded {len(imagenet_val)} images from {len(imagenet_val.classes)} classes")

In [ ]:
# Cell 5: Load models
import torchvision.models as models
from robustbench.utils import load_model

print("Loading vanilla ResNet50...")
vanilla = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
vanilla = vanilla.to(device).eval()

print("Loading adversarially-trained model (Salman2020Do_R50) from RobustBench...")
robust = load_model(
    model_name='Salman2020Do_R50',
    dataset='imagenet',
    threat_model='Linf'
)
robust = robust.to(device).eval()

print("Models loaded!")

In [ ]:
# Cell 6: Get mutually correct samples
NUM_SAMPLES = 500

def get_mutually_correct_samples(model1, model2, dataset, n=500):
    """Get samples correctly classified by BOTH models."""
    model1.eval()
    model2.eval()
    
    samples = []
    
    with torch.no_grad():
        for i in range(len(dataset)):
            if len(samples) >= n:
                break
            
            img, label = dataset[i]
            img_batch = img.unsqueeze(0).to(device)
            
            pred1 = model1(img_batch).argmax().item()
            pred2 = model2(img_batch).argmax().item()
            
            # Both models must predict correctly
            if pred1 == label and pred2 == label:
                samples.append((img, label))
            
            if i % 200 == 0:
                print(f"  Scanned {i} images, found {len(samples)} mutual correct...")
    
    return samples

print(f"Finding {NUM_SAMPLES} samples correctly classified by both models...")
samples = get_mutually_correct_samples(vanilla, robust, imagenet_val, n=NUM_SAMPLES)
print(f"\nFound {len(samples)} mutually correct samples")

In [ ]:
# Cell 7: Test 1 - Natural Perturbations
from visprobe import search

results = {"vanilla": {}, "robust": {}}

print("=" * 60)
print("TEST 1: NATURAL PERTURBATIONS (Gaussian Blur)")
print("=" * 60)

for model_name, model in [("vanilla", vanilla), ("robust", robust)]:
    print(f"\nTesting {model_name}...")
    report = search(
        model=model,
        data=samples,
        perturbation="gaussian_blur",
        level_lo=0.0,
        level_hi=10.0,
        device=device,
        normalization="imagenet"
    )
    results[model_name]["natural"] = report.score
    print(f"  {model_name}: {report.score:.1f}% robust")

In [ ]:
# Cell 8: Test 2 - Adversarial Attacks (PGD)
from visprobe.strategies import PGDStrategy

print("\n" + "=" * 60)
print("TEST 2: ADVERSARIAL ATTACKS (PGD)")
print("This is what adversarial training was designed for")
print("=" * 60)

for model_name, model in [("vanilla", vanilla), ("robust", robust)]:
    print(f"\nTesting {model_name}...")
    report = search(
        model=model,
        data=samples,
        strategy=lambda eps: PGDStrategy(eps=eps, eps_step=max(eps/10, 1e-6), max_iter=20),
        level_lo=0.0,
        level_hi=0.03,
        device=device,
        normalization="imagenet",
        strategy_name="PGD Attack"
    )
    results[model_name]["adversarial"] = report.score
    print(f"  {model_name}: {report.score:.1f}% robust")

In [ ]:
# Cell 9: Test 3 - Compositional Attacks (THE KEY CONTRIBUTION)
from visprobe.strategies import GaussianBlurStrategy, BrightnessStrategy, GaussianNoiseStrategy, Compose

print("\n" + "=" * 60)
print("TEST 3: COMPOSITIONAL ATTACKS (Novel Threat Model)")
print("Natural perturbation + Adversarial attack combined")
print("=" * 60)

scenarios = [
    (
        "Low-Light PGD",
        lambda s: Compose([
            BrightnessStrategy(brightness_factor=0.3 + 0.7 * (1 - s)),  # Darken image
            PGDStrategy(eps=s * 0.01, eps_step=max(s * 0.001, 1e-6), max_iter=10)
        ])
    ),
    (
        "Blurred PGD",
        lambda s: Compose([
            GaussianBlurStrategy(sigma=s * 2.0),  # Blur image
            PGDStrategy(eps=s * 0.01, eps_step=max(s * 0.001, 1e-6), max_iter=10)
        ])
    ),
    (
        "Noisy PGD",
        lambda s: Compose([
            GaussianNoiseStrategy(std_dev=s * 0.03),  # Add noise
            PGDStrategy(eps=s * 0.01, eps_step=max(s * 0.001, 1e-6), max_iter=10)
        ])
    ),
]

for scenario_name, composition in scenarios:
    print(f"\n{scenario_name}:")
    
    for model_name, model in [("vanilla", vanilla), ("robust", robust)]:
        report = search(
            model=model,
            data=samples,
            strategy=composition,
            level_lo=0.0,
            level_hi=1.0,
            device=device,
            normalization="imagenet",
            strategy_name=scenario_name
        )
        results[model_name][scenario_name] = report.score
        print(f"  {model_name}: {report.score:.1f}% robust")

In [ ]:
# Cell 10: Final Results Table
print("\n" + "=" * 80)
print("ADVERSARIAL TRAINING EFFECTIVENESS - FINAL RESULTS")
print("Dataset: ImageNet validation set")
print(f"Samples: {len(samples)} (correctly classified by both models)")
print("=" * 80)

print("\nThreat Model            | Vanilla | Adv-Trained | Improvement")
print("-" * 65)

# Natural
v_nat = results["vanilla"]["natural"]
r_nat = results["robust"]["natural"]
print(f"Natural (Blur)          | {v_nat:6.1f}% |    {r_nat:6.1f}% |  {r_nat-v_nat:+.1f}%")

# Adversarial
v_adv = results["vanilla"]["adversarial"]
r_adv = results["robust"]["adversarial"]
print(f"Adversarial (PGD)       | {v_adv:6.1f}% |    {r_adv:6.1f}% |  {r_adv-v_adv:+.1f}%")

print("\nCompositional (Novel):")
for scenario in ["Low-Light PGD", "Blurred PGD", "Noisy PGD"]:
    v = results["vanilla"][scenario]
    r = results["robust"][scenario]
    improvement = r - v
    status = "PROTECTED" if improvement >= 10 else "VULNERABLE"
    print(f"  {scenario:20}  | {v:6.1f}% |    {r:6.1f}% |  {improvement:+.1f}% [{status}]")

print("\n" + "=" * 80)
print("KEY FINDING:")
print("  - Adversarial training provides strong improvement on pure PGD attacks")
print("  - BUT shows limited improvement on compositional attacks")
print("  - Standard adversarial training is INSUFFICIENT for real-world robustness!")
print("=" * 80)

In [ ]:
# Cell 11: Save and download results
import json
from google.colab import files

# Save full results
experiment_data = {
    "config": {
        "dataset": "ImageNet",
        "num_samples": len(samples),
        "vanilla_model": "ResNet50 (IMAGENET1K_V2)",
        "robust_model": "Salman2020Do_R50 (RobustBench)",
        "device": device
    },
    "results": results
}

with open('imagenet_experiment_results.json', 'w') as f:
    json.dump(experiment_data, f, indent=2)

files.download('imagenet_experiment_results.json')
print("Results downloaded!")

In [ ]:
# Cell 12: Generate visualization (optional)
import matplotlib.pyplot as plt
import numpy as np

# Prepare data for plotting
categories = ['Natural\n(Blur)', 'Adversarial\n(PGD)', 'Low-Light\nPGD', 'Blurred\nPGD', 'Noisy\nPGD']
vanilla_scores = [
    results["vanilla"]["natural"],
    results["vanilla"]["adversarial"],
    results["vanilla"]["Low-Light PGD"],
    results["vanilla"]["Blurred PGD"],
    results["vanilla"]["Noisy PGD"]
]
robust_scores = [
    results["robust"]["natural"],
    results["robust"]["adversarial"],
    results["robust"]["Low-Light PGD"],
    results["robust"]["Blurred PGD"],
    results["robust"]["Noisy PGD"]
]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, vanilla_scores, width, label='Vanilla ResNet50', color='#ff6b6b')
bars2 = ax.bar(x + width/2, robust_scores, width, label='Adversarially-Trained', color='#4ecdc4')

ax.set_ylabel('Robustness Score (%)', fontsize=12)
ax.set_xlabel('Threat Model', fontsize=12)
ax.set_title('Adversarial Training Effectiveness on ImageNet\n(Higher = More Robust)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.set_ylim(0, 100)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

# Add dividing line between standard and compositional
ax.axvline(x=1.5, color='gray', linestyle='--', alpha=0.5)
ax.text(0.5, 95, 'Standard Threats', ha='center', fontsize=10, color='gray')
ax.text(3, 95, 'Compositional Threats', ha='center', fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('adversarial_training_effectiveness.png', dpi=150, bbox_inches='tight')
plt.show()

files.download('adversarial_training_effectiveness.png')
print("Figure downloaded!")